# Grad-CAM Visualization for KGCL

This notebook loads a trained `ISICImageOnly` or `MGCA_ISIC` checkpoint and visualizes a Grad-CAM heatmap for a single dermoscopy image.

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve().parent
KGCL_ROOT = PROJECT_ROOT / "KGCL"
for path in (PROJECT_ROOT, KGCL_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from KGCL.models.kgcl.image_module import ISICImageOnly
from KGCL.models.kgcl.kgcl_module import MGCA_ISIC
from KGCL.datasets.transforms import DataTransforms
from KGCL.datasets.constants import IDX_TO_DIAGNOSIS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Update these paths for your run.
CHECKPOINT_PATH = PROJECT_ROOT / "KGCL" / "checkpoints" / "joint" / "YOUR_RUN" / "last.ckpt"
IMAGE_PATH = PROJECT_ROOT / "Dataset" / "Images" / "ISIC_0000000.jpg"
MODEL_KIND = "joint"  # "joint" or "image_only"
IMG_ENCODER = "vit_base"  # "resnet_50" or "vit_base"
TARGET_CLASS = None  # None uses predicted class. Set 0 for NV or 1 for MEL.

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert IMAGE_PATH.exists(), f"Image not found: {IMAGE_PATH}"

In [ ]:
def build_model(model_kind: str, img_encoder: str):
    if model_kind == "joint":
        model = MGCA_ISIC.load_from_checkpoint(
            str(CHECKPOINT_PATH),
            map_location=device,
            img_encoder=img_encoder,
            freeze_bert=True,
        )
    elif model_kind == "image_only":
        model = ISICImageOnly.load_from_checkpoint(
            str(CHECKPOINT_PATH),
            map_location=device,
            img_encoder=img_encoder,
        )
    else:
        raise ValueError(f"Unsupported MODEL_KIND: {model_kind}")
    model.eval()
    model.to(device)
    return model


def get_target_module(model, img_encoder: str):
    if "resnet" in img_encoder:
        return model.img_encoder_q.model.layer4[-1]
    if "vit" in img_encoder:
        return model.img_encoder_q.model.blocks[-1].norm1
    raise ValueError(f"Unsupported encoder: {img_encoder}")


def forward_diagnosis_logits(model, imgs: torch.Tensor) -> torch.Tensor:
    img_feat_q, _ = model.img_encoder_q(imgs)
    if img_feat_q.ndim == 4:
        img_feat_raw = F.adaptive_avg_pool2d(img_feat_q, 1).flatten(1)
    elif img_feat_q.ndim == 3:
        img_feat_raw = img_feat_q[:, 0]
    else:
        img_feat_raw = img_feat_q
    return model.diagnosis_head(img_feat_raw)


model = build_model(MODEL_KIND, IMG_ENCODER)
target_module = get_target_module(model, IMG_ENCODER)
model

In [ ]:
transform = DataTransforms(is_train=False, crop_size=224)

pil_image = Image.open(IMAGE_PATH).convert("RGB")
input_tensor = transform(pil_image).unsqueeze(0).to(device)

display(pil_image)
print("Input tensor shape:", tuple(input_tensor.shape))

In [ ]:
class GradCAM:
    def __init__(self, model, target_module, img_encoder: str):
        self.model = model
        self.target_module = target_module
        self.img_encoder = img_encoder
        self.activations = None
        self.gradients = None
        self.forward_handle = target_module.register_forward_hook(self._save_activation)
        self.backward_handle = target_module.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def __call__(self, x: torch.Tensor, target_class: int | None = None):
        self.model.zero_grad(set_to_none=True)
        logits = forward_diagnosis_logits(self.model, x)
        pred_class = int(logits.argmax(dim=1).item())
        class_idx = pred_class if target_class is None else int(target_class)
        score = logits[:, class_idx].sum()
        score.backward(retain_graph=True)

        if "resnet" in self.img_encoder:
            activations = self.activations
            gradients = self.gradients
            weights = gradients.mean(dim=(2, 3), keepdim=True)
            cam = (weights * activations).sum(dim=1, keepdim=True)
        else:
            activations = self.activations[:, 1:, :]
            gradients = self.gradients[:, 1:, :]
            side = int(activations.shape[1] ** 0.5)
            weights = gradients.mean(dim=1, keepdim=True)
            cam = (weights * activations).sum(dim=2).reshape(-1, 1, side, side)

        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        probs = logits.softmax(dim=1).detach().cpu().numpy()[0]
        return cam, pred_class, class_idx, probs


In [ ]:
def denormalize_image(tensor: torch.Tensor) -> np.ndarray:
    image = tensor.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    image = (image * 0.5) + 0.5
    return np.clip(image, 0.0, 1.0)


def overlay_heatmap(image: np.ndarray, cam: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    heatmap = np.uint8(255 * cam)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    overlay = np.clip((1 - alpha) * image + alpha * heatmap, 0.0, 1.0)
    return overlay


grad_cam = GradCAM(model, target_module, IMG_ENCODER)
cam, pred_class, used_class, probs = grad_cam(input_tensor, TARGET_CLASS)
base_image = denormalize_image(input_tensor)
overlay = overlay_heatmap(base_image, cam)

print(f"Predicted class: {pred_class} ({IDX_TO_DIAGNOSIS[pred_class]})")
print(f"Grad-CAM target class: {used_class} ({IDX_TO_DIAGNOSIS[used_class]})")
print(f"Class probabilities: NV={probs[0]:.4f}, MEL={probs[1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(base_image)
axes[0].set_title("Preprocessed image")
axes[0].axis("off")

axes[1].imshow(cam, cmap="jet")
axes[1].set_title("Grad-CAM heatmap")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title(f"Overlay ({IDX_TO_DIAGNOSIS[used_class]})")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Run this when you are finished with the notebook session.
grad_cam.remove()